In [1]:
import os
import random
import warnings
import traceback
from pathlib import Path
from typing import Dict, Optional, Tuple
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import quad4fhe

from plain_experiment_io import (
    build_quad4fhe_run_record,
    make_experiment_payload,
    save_csv,
    save_json,
    tee_stdout_stderr,
    write_combined_dataset_json,
)


# ============================================================
# Global Configuration
# ============================================================
SEED = 2026
HIDDEN_DIM = 256
EPOCHS = 200
LR = 1e-3

EARLY_STOP_PATIENCE = 20
EARLY_STOP_MIN_DELTA = 1e-5

# Data split fractions
TEST_SIZE = 0.20
TRAIN_NET_SIZE = 0.50
VAL_NET_SIZE = 0.10
REPLACEMENT_SELECTION_FRACTION = 0.5

# Pool sizes to sweep
POOL_SIZES_TO_COMPARE = [0.01, 0.05, 0.10, 0.20]

DIABETES_CSV_PATH = "diabetes_dataset.csv"
DIABETES_TARGET_COL = None

BASELINES = ["square", "ls_poly_2", "ls_poly_3", "ls_poly_5", "ls_poly_7",
             "remez_2", "remez_3", "remez_5", "remez_7",
             "ola", "precise_a11"]

# ============================================================
# Automatic result export
# ============================================================
DATASET_NAME = "Diabetes"
EXPERIMENT_NAME = "smallpool"
OUTPUT_DIR = Path("..") / "results" / DATASET_NAME / EXPERIMENT_NAME
LOG_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_result.txt"
JSON_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_results.json"
META_CSV_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_meta.csv"


# ============================================================
# Decision-preservation diagnostics
# ============================================================
def _quad_report_diagnostics(result, split: str, n_expected: int = None):
    """Return calibration/test agreement and mismatch count for Quad4FHE.

    Newer quad4fhe versions expose result.fit_diagnostics/result.test_diagnostics.
    The fallback below extracts the same information from report_vs_pseudo.
    """
    attr_name = "fit_diagnostics" if split == "fit" else "test_diagnostics"
    diag = getattr(result, attr_name, None)
    if diag is not None:
        return dict(diag)

    df = getattr(result, "report_vs_pseudo", None)
    if df is None:
        return {
            "split": split, "n": n_expected, "agreement": None,
            "mismatch_count": None, "exact_preserved": None,
        }
    row = df[(df["Method"] == "Quad4FHE") & (df["Split"] == split)]
    if len(row) == 0:
        return {
            "split": split, "n": n_expected, "agreement": None,
            "mismatch_count": None, "exact_preserved": None,
        }
    agreement = float(row.iloc[0]["Agreement"])
    mismatch = None if n_expected is None else int(round((1.0 - agreement) * int(n_expected)))
    return {
        "split": split,
        "n": n_expected,
        "agreement": agreement,
        "mismatch_count": mismatch,
        "exact_preserved": None if mismatch is None else bool(mismatch == 0),
    }


def _quad_solver_diagnostics(result):
    """Return margin/slack diagnostics when the quad4fhe library exposes them."""
    return dict(getattr(result, "solver_diagnostics", None) or {})

# ============================================================
# Model + EarlyStopping
# ============================================================
class SingleHiddenLayerMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x))).squeeze(-1)


class EarlyStopping:
    def __init__(self, patience=EARLY_STOP_PATIENCE, min_delta=EARLY_STOP_MIN_DELTA):
        self.patience, self.min_delta = patience, min_delta
        self.best_loss = float("inf"); self.counter = 0
        self.best_epoch = 0; self.best_state = None

    def step(self, val_loss, epoch, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss, self.counter, self.best_epoch = val_loss, 0, epoch
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def train_relu_mlp(X_tr, y_tr, X_va, y_va, input_dim, hidden_dim):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SingleHiddenLayerMLP(input_dim, hidden_dim).to(device)

    X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32).to(device)
    X_va_t = torch.tensor(X_va, dtype=torch.float32).to(device)
    y_va_t = torch.tensor(y_va, dtype=torch.float32).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)
    bce = nn.BCEWithLogitsLoss()
    stopper = EarlyStopping()

    print(f"  Architecture: Linear({input_dim}->{hidden_dim}) -> ReLU -> Linear({hidden_dim}->1)")
    print(f"  Device={device}, epochs={EPOCHS}, lr={LR}")

    for epoch in range(EPOCHS):
        model.train()
        opt.zero_grad()
        train_loss = bce(model(X_tr_t), y_tr_t)
        train_loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = bce(model(X_va_t), y_va_t).item()

        if (epoch + 1) % 50 == 0:
            print(f"    epoch {epoch+1:>4d}  train_loss={train_loss.item():.6f}  val_loss={val_loss:.6f}")

        if stopper.step(val_loss, epoch + 1, model):
            print(f"  Early stopping @ epoch {epoch + 1} "
                  f"(best val_loss={stopper.best_loss:.6f} @ epoch {stopper.best_epoch})")
            stopper.restore(model)
            break
    else:
        stopper.restore(model)
    model.cpu(); model.eval()
    return model


# ============================================================
# Data loading
# ============================================================
def load_diabetes_csv(csv_path, target_col=None):
    df = pd.read_csv(csv_path)
    print(f"Loaded CSV: {csv_path}  shape={df.shape}")

    if target_col is not None and target_col in df.columns:
        tgt = target_col
    else:
        candidates = ['Outcome', 'outcome', 'Diabetes', 'diabetes',
                       'Target', 'target', 'Class', 'class', 'Label', 'label',
                       'Diabetic', 'diabetic']
        tgt = next((c for c in candidates if c in df.columns), df.columns[-1])
    print(f"  Target column: '{tgt}'")

    y_all = df[tgt].values.astype(int)
    feature_cols = [c for c in df.columns if c != tgt]
    X_all = df[feature_cols].values.astype(np.float64)

    if np.isnan(X_all).any():
        medians = np.nanmedian(X_all, axis=0)
        for j in range(X_all.shape[1]):
            mask = np.isnan(X_all[:, j])
            X_all[mask, j] = medians[j]

    uniq = np.unique(y_all)
    if not np.array_equal(uniq, np.array([0, 1])):
        y_all = (y_all == uniq[1]).astype(int)

    print(f"  samples={len(y_all)}, features={len(feature_cols)}, "
          f"pos={int(np.sum(y_all == 1))}, neg={int(np.sum(y_all == 0))}")
    return X_all, y_all


# ============================================================
# Fixed-ratio splitting with variable pool size
# ============================================================
def split_dataset_fixed(X, y, pool_frac):
    # 1) Carve out test (20%)
    X_rest, X_test, y_rest, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)

    # 2) Carve out val_net
    val_frac_in_rest = VAL_NET_SIZE / (1.0 - TEST_SIZE)
    X_rest2, X_val_net, y_rest2, y_val_net = train_test_split(
        X_rest, y_rest, test_size=val_frac_in_rest,
        random_state=SEED + 1, stratify=y_rest)

    # 3) Carve out train_net and pool_max
    train_frac_in_rest2 = TRAIN_NET_SIZE / (1.0 - TEST_SIZE - VAL_NET_SIZE)
    X_train_net, X_pool_max, y_train_net, y_pool_max = train_test_split(
        X_rest2, y_rest2, test_size=1.0 - train_frac_in_rest2,
        random_state=SEED + 2, stratify=y_rest2)

    # 4) Sub-sample pool_max to get desired pool_frac
    max_pool_frac = 1.0 - TEST_SIZE - VAL_NET_SIZE - TRAIN_NET_SIZE
    if pool_frac >= max_pool_frac - 1e-6:
        X_pool, y_pool = X_pool_max, y_pool_max
    else:
        keep = pool_frac / max_pool_frac
        X_pool, _, y_pool, _ = train_test_split(
            X_pool_max, y_pool_max, train_size=keep,
            random_state=SEED + 3, stratify=y_pool_max)

    # 5) Split pool into rep_fit / rep_sel (50/50)
    X_rep_fit, X_rep_sel, y_rep_fit, y_rep_sel = train_test_split(
        X_pool, y_pool, test_size=REPLACEMENT_SELECTION_FRACTION,
        random_state=SEED + 4, stratify=y_pool)

    X_train_main = np.vstack([X_train_net, X_val_net])
    y_train_main = np.concatenate([y_train_net, y_val_net])

    return {
        "train_net": (X_train_net, y_train_net),
        "val_net":   (X_val_net, y_val_net),
        "train_main":(X_train_main, y_train_main),
        "rep_fit":   (X_rep_fit, y_rep_fit),
        "rep_sel":   (X_rep_sel, y_rep_sel),
        "test":      (X_test, y_test),
    }


# ============================================================
# Extract test-set metrics from ReplacementResult
# ============================================================
def extract_test_metrics(result: quad4fhe.ReplacementResult,
                         method_name: str) -> Optional[Dict]:
    """Pull the 'test' split row for `method_name` out of report_vs_truth."""
    if result.report_vs_truth is None:
        return None
    df = result.report_vs_truth
    row = df[(df["Method"] == method_name) & (df["Split"] == "test")]
    if len(row) == 0:
        return None
    return row.iloc[0].to_dict()


# ============================================================
# Main
# ============================================================
def main():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    # ==========================================================
    print("\n" + "=" * 78)
    print("Step 1: Load diabetes dataset")
    print("=" * 78)
    X_all, y_all = load_diabetes_csv(DIABETES_CSV_PATH, DIABETES_TARGET_COL)
    d = X_all.shape[1]

    print(f"\nFixed splits: train_net={TRAIN_NET_SIZE*100:.0f}%, "
          f"val_net={VAL_NET_SIZE*100:.0f}%, test={TEST_SIZE*100:.0f}%")
    print(f"Pool sizes to compare: {[f'{p*100:.0f}%' for p in POOL_SIZES_TO_COMPARE]}")

    # ==========================================================
    print("\n" + "=" * 78)
    print("Step 2: Create fixed splits (train_net / val_net / test are fixed across pools)")
    print("=" * 78)
    max_pool = max(POOL_SIZES_TO_COMPARE)
    splits = split_dataset_fixed(X_all, y_all, pool_frac=max_pool)

    X_train_net_raw, y_train_net = splits["train_net"]
    X_val_net_raw,   y_val_net   = splits["val_net"]
    X_test_raw,      y_test      = splits["test"]

    scaler = StandardScaler().fit(X_train_net_raw)
    X_train_net = scaler.transform(X_train_net_raw).astype(np.float64)
    X_val_net   = scaler.transform(X_val_net_raw).astype(np.float64)
    X_test      = scaler.transform(X_test_raw).astype(np.float64)

    for key in ["train_net", "val_net", "train_main", "rep_fit", "rep_sel", "test"]:
        Xi, yi = splits[key]
        print(f"  {key:12s}: n={len(Xi):6d}")

    payload = make_experiment_payload(
        dataset=DATASET_NAME,
        experiment=EXPERIMENT_NAME,
        seed=SEED,
        dataset_info={
            "input_dim": int(d),
            "num_classes": 2,
            "splits": {key: len(splits[key][0]) for key in ["train_net", "val_net", "train_main", "rep_fit", "rep_sel", "test"]},
            "pool_subsets": [],
            "split_protocol": "single fixed stratified split",
        },
        config={
            "hidden_dim": HIDDEN_DIM,
            "epochs": EPOCHS,
            "lr": LR,
            "test_size": TEST_SIZE,
            "train_net_size": TRAIN_NET_SIZE,
            "val_net_size": VAL_NET_SIZE,
            "replacement_selection_fraction": REPLACEMENT_SELECTION_FRACTION,
            "pool_sizes_to_compare": POOL_SIZES_TO_COMPARE,
            "baselines": BASELINES,
            "early_stop_patience": EARLY_STOP_PATIENCE,
            "early_stop_min_delta": EARLY_STOP_MIN_DELTA,
        },
        source_script=Path(__file__).name if "__file__" in globals() else "notebook",
        log_file=LOG_PATH,
    )

    # ==========================================================
    print("\n" + "=" * 78)
    print("Step 3: Train the ReLU MLP once (shared across all pool sizes)")
    print("=" * 78)
    model = train_relu_mlp(X_train_net, y_train_net, X_val_net, y_val_net, d, HIDDEN_DIM)

    # Original test-set accuracy for reference
    with torch.no_grad():
        relu_logits_test = model(torch.tensor(X_test, dtype=torch.float32)).cpu().numpy()
    orig_test_acc = float(np.mean((relu_logits_test > 0.0) == y_test))
    print(f"\n  Original model test ACC = {orig_test_acc:.4f}")

    # ==========================================================
    print("\n" + "=" * 78)
    print("Step 4: Call quad4fhe.replace(...) once per pool size")
    print("=" * 78)
    all_pool_results = {}

    for pool_frac in POOL_SIZES_TO_COMPARE:
        print("\n" + "-" * 78)
        print(f"POOL = {pool_frac*100:.0f}%")
        print("-" * 78)

        # Re-split with this pool size — train_net/val_net/test are IDENTICAL,
        # only rep_fit / rep_sel change.
        splits_p = split_dataset_fixed(X_all, y_all, pool_frac=pool_frac)
        X_rep_fit_raw, y_rep_fit = splits_p["rep_fit"]
        X_rep_sel_raw, y_rep_sel = splits_p["rep_sel"]

        X_rep_fit = scaler.transform(X_rep_fit_raw).astype(np.float64)
        X_rep_sel = scaler.transform(X_rep_sel_raw).astype(np.float64)

        print(f"  rep_fit: n={len(X_rep_fit)},  rep_sel: n={len(X_rep_sel)}")

        # Use rep_fit to fit the quadratic, rep_sel as held-out evaluation.
        # test_data=(X_test, y_test) produces the evaluation tables.
        result = quad4fhe.replace(
            task          = "binary",
            model         = model,
            hidden_layer  = "fc1",
            output_layer  = "fc2",
            activation    = "relu",
            fit_data      = (X_rep_fit, y_rep_fit),
            test_data     = (X_test, y_test),
            baselines     = BASELINES,
            export_he_dir = f"he_artifacts_pool_{int(pool_frac*100):02d}",
            use_cutting_plane = False,
            use_osqp_warmstart= False,
            verbose       = True,
            seed          = SEED,
        )
        all_pool_results[pool_frac] = result
        payload["dataset_info"]["pool_subsets"].append({
            "fraction": pool_frac,
            "rep_fit_size": len(X_rep_fit),
            "rep_sel_size": len(X_rep_sel),
        })
        payload["runs"].append(build_quad4fhe_run_record(
            result=result,
            key=f"pool={int(pool_frac * 100):02d}%",
            hidden_dim=HIDDEN_DIM,
            fit_n=len(X_rep_fit),
            test_n=len(X_test),
            pool_fraction=pool_frac,
            rep_fit_size=len(X_rep_fit),
        ))
        # Save a checkpoint after each pool, so a long experiment still leaves
        # a usable JSON if a later pool fails.
        save_json(payload, JSON_PATH)

        # Print a per-pool one-liner summary
        fit_diag = _quad_report_diagnostics(result, "fit", n_expected=len(X_rep_fit))
        test_diag = _quad_report_diagnostics(result, "test", n_expected=len(X_test))
        print(f"  method_used={result.method_used}, "
              f"hard_feasible={result.feasible}, "
              f"quant_decimals={result.quant_decimals}")
        print(f"  calib agreement={fit_diag.get('agreement')} "
              f"mismatches={fit_diag.get('mismatch_count')} "
              f"exact_preserved={fit_diag.get('exact_preserved')}")
        q4_row = extract_test_metrics(result, "Quad4FHE")
        if q4_row is not None:
            print(f"  Quad4FHE test ACC={q4_row['ACC']:.4f}  AUC={q4_row['AUC']:.4f}  "
                  f"F1={q4_row['F1']:.4f}  Agreement={q4_row['Agreement']:.4f}  "
                  f"test_mismatches={test_diag.get('mismatch_count')}")

    # ==========================================================
    print("\n" + "=" * 78)
    print("Step 5: Cross-pool comparison tables")
    print("=" * 78)

    # All the method names that should appear in a report (must match the
    # names the library uses: "Original", "Quad4FHE", and baseline names
    # lowercased).
    method_order = ["Original", "Quad4FHE"] + BASELINES

    pool_labels = [f"{int(p*100):02d}%" for p in POOL_SIZES_TO_COMPARE]

    def _build_matrix(metric_key: str) -> pd.DataFrame:
        """Return a DataFrame with rows=method, columns=pool_frac."""
        data = {}
        for pool_frac, lbl in zip(POOL_SIZES_TO_COMPARE, pool_labels):
            result = all_pool_results[pool_frac]
            col = {}
            for m in method_order:
                row = extract_test_metrics(result, m)
                col[m] = row[metric_key] if row is not None else float("nan")
            data[f"pool={lbl}"] = col
        return pd.DataFrame(data)

    cross_pool_tables = {}
    for metric_key in ["ACC", "AUC", "F1", "Precision", "Recall", "Agreement"]:
        title = ("Test Agreement (with original model / pseudo-labels)"
                 if metric_key == "Agreement"
                 else f"Test {metric_key} (vs TRUE labels)")
        print(f"\n--- Table: {title} ---")
        df = _build_matrix(metric_key)
        cross_pool_tables[metric_key] = df.to_dict()
        with pd.option_context("display.float_format", "{:.4f}".format):
            print(df.to_string())

    payload["cross_pool_tables"] = cross_pool_tables

    # --- Meta table: feasibility, method_used, quant_decimals ---
    print("\n--- Meta Table ---")
    meta_rows = []
    for pool_frac, lbl in zip(POOL_SIZES_TO_COMPARE, pool_labels):
        r = all_pool_results[pool_frac]
        fit_diag = _quad_report_diagnostics(r, "fit", n_expected=len(split_dataset_fixed(X_all, y_all, pool_frac=pool_frac)["rep_fit"][0]))
        test_diag = _quad_report_diagnostics(r, "test", n_expected=len(X_test))
        solver_diag = _quad_solver_diagnostics(r)
        meta_rows.append({
            "pool":             lbl,
            "method_used":      r.method_used,
            "hard_feasible":    r.feasible,
            "empirical_margin": round(r.empirical_margin, 6),
            "norm_margin":      round(r.normalized_margin, 6),
            "quant_decimals":   r.quant_decimals,
            "calib_n":          fit_diag.get("n"),
            "calib_agreement":  fit_diag.get("agreement"),
            "calib_mismatch":   fit_diag.get("mismatch_count"),
            "exact_preserved":  fit_diag.get("exact_preserved"),
            "test_agreement":   test_diag.get("agreement"),
            "test_mismatch":    test_diag.get("mismatch_count"),
            "calib_test_gap":   (None if fit_diag.get("agreement") is None or test_diag.get("agreement") is None
                                  else fit_diag.get("agreement") - test_diag.get("agreement")),
            "slack_pos":        solver_diag.get("slack_positive_count"),
            "sum_slack":        solver_diag.get("sum_slack"),
            "max_slack":        solver_diag.get("max_slack"),
            "constraint_version": getattr(r, "constraint_version", None),
            "he_export_dir":    os.path.basename(r.he_export_dir) if r.he_export_dir else None,
        })
    meta_df = pd.DataFrame(meta_rows)
    with pd.option_context("display.float_format", "{:.6f}".format):
        print(meta_df.to_string(index=False))

    payload["meta_table"] = meta_rows
    save_json(payload, JSON_PATH)
    save_csv(meta_rows, META_CSV_PATH)
    write_combined_dataset_json(DATASET_NAME, root_dir=OUTPUT_DIR.parent.parent)

    print("\n" + "=" * 78)
    print("Done.")
    print("=" * 78)


if __name__ == "__main__":
    try:
        with tee_stdout_stderr(LOG_PATH):
            main()
    except Exception:
        # The traceback is written both to screen and to the autosaved log.
        traceback.print_exc()
        raise

[autosave] stdout/stderr log -> ..\results\Diabetes\smallpool\Diabetes_smallpool_result.txt

Step 1: Load diabetes dataset
Loaded CSV: diabetes_dataset.csv  shape=(9538, 17)
  Target column: 'Outcome'
  samples=9538, features=16, pos=3282, neg=6256

Fixed splits: train_net=50%, val_net=10%, test=20%
Pool sizes to compare: ['1%', '5%', '10%', '20%']

Step 2: Create fixed splits (train_net / val_net / test are fixed across pools)
  train_net   : n=  4768
  val_net     : n=   954
  train_main  : n=  5722
  rep_fit     : n=   954
  rep_sel     : n=   954
  test        : n=  1908

Step 3: Train the ReLU MLP once (shared across all pool sizes)
  Architecture: Linear(16->256) -> ReLU -> Linear(256->1)
  Device=cuda, epochs=200, lr=0.001
    epoch   50  train_loss=0.294348  val_loss=0.294250
    epoch  100  train_loss=0.112198  val_loss=0.119403
    epoch  150  train_loss=0.063270  val_loss=0.072458
    epoch  200  train_loss=0.045068  val_loss=0.054209

  Original model test ACC = 0.9885

Ste